# Part 1: Build a multi-source knowledge base

In Foundry IQ (Azure AI Search), a **knowledge base** is a top-level resource that connects a chat completion model to one or more knowledge sources (searchable indexes or remote data sources). It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized. Once created, you can update it at any time.

## Pre-configured environment

This lab uses the following Azure resources, already set up for you:

* Azure AI Search
  * `hrdocs` index: HR policies, handbook, company info
  * `healthdocs` index: health benefits, insurance, coverage
  * Semantic ranking enabled, pre-computed embeddings
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-4.1`: Chat completion for reasoning and synthesis
  * `text-embedding-3-large`: Embedding model for vector search
* Microsoft Fabric

## Step 1: Setup environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## Step 2: Verify search indexes

As part of the lab setup, the search indexes were pre-populated with ingested document chunks. Run the cell below to confirm each index has the expected number of document chunks:

* `hrdocs`: 50 document chunks about HR policies
* `healthdocs` index: 334 document chunks about health benefits and insurance

In [ ]:
from azure.search.documents import SearchClient

for index_name in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, index_name, search_credential)
    count = client.get_document_count()
    print(f"{index_name}: {count} documents")


## Step 3: Create two knowledge sources

A **knowledge source** connects your knowledge base to actual data. You'll create two knowledge sources pointing to each of those indexes.

* `healthdocs-knowledge-source`: Points to the `healthdocs` index
* `hrdocs-knowledge-source`: Points to the `hrdocs` index

Both sources set these fields as the searchable fields:

* `snippet`: The actual text content
* `blob_path`: Where the original document lives (used for citation links)


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "HR documents from the restored hrdocs index"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Health benefits documents from the restored healthdocs index"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")


## Step 4: Create the multi-source knowledge base

A knowledge base is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 2)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

For this notebook, we configure the knowledge base to use an output mode of `extractiveData`, which means it will return the merged results for the query. It's also possible to use `answerSynthesis` to use the attached LLM to also generate the complete answer.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeSourceReference,
)

KNOWLEDGE_BASE_NAME = "multisource-search-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base over HR and health document indexes",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")


## Step 5: Query the knowledge base

Let's ask a two-part question that spans both knowledge sources:

* "What is the responsibility of the Zava CEO?" (HR-related)
* "What health plan would you recommend for the best mental health coverage?" (Health-related)

The knowledge base uses agentic retrieval to:

1. Analyze the query to understand two different topics
2. Decompose the query into focused subqueries
3. Determine which knowledge sources are relevant for each subquery
4. Run searches concurrently against the selected sources
5. Merge the results with the semanting re-ranking model

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(
                    text="What is the responsibility of the Zava CEO? "
                    "What health plan would you recommend if they wanted the best coverage for mental health services?"
                )
            ],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
activity_by_id = {activity.id: getattr(activity, "knowledge_source_name", None) for activity in (result.activity or [])}
references_json = []

for reference in result.references or []:
    reference_json = reference.as_dict()
    reference_json["knowledgeSourceName"] = activity_by_id.get(reference.activity_source, "unknown")
    references_json.append(reference_json)

print(json.dumps(references_json, indent=2))

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

## Summary

You've completed Part 1! You created two indexed knowledge sources, built a multi-source knowledge base with Azure OpenAI, and queried using agentic retrieval with citation-backed answers.

Key concepts:

* A single knowledge base can reference multiple knowledge sources
* The knowledge base intelligently selects which sources to query based on the question
* Activity logs show how the agent processed your query

➡️ Continue to [Part 2: Add grounding from web results](part2-mai-grounding-mcp-kb.ipynb).